In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

DATASET_PATH = "/content/drive/MyDrive/fundus/colored_images"

assert os.path.exists(DATASET_PATH), f"Dataset not found: {DATASET_PATH}"

classes = sorted([
    name for name in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, name))
])

print("Dataset path verified!")
print("Number of classes:", len(classes))
print("Classes:", classes)

for class_name in classes:
    folder = os.path.join(DATASET_PATH, class_name)
    images = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"))
    ]
    print(f"{class_name}: {len(images)} images")

Dataset path verified!
Number of classes: 5
Classes: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']
Mild: 370 images
Moderate: 999 images
No_DR: 1805 images
Proliferate_DR: 295 images
Severe: 193 images


In [3]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU detected. Enable it via Runtime → Change runtime type → T4 GPU.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [4]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)

from torchvision.models import resnet18, ResNet18_Weights

print("ResNet18 imported successfully!")

PyTorch: 2.11.0+cu130
TorchVision: 0.26.0+cu130
ResNet18 imported successfully!


In [5]:
import os
import random
import json
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Paths
DATASET_PATH = "/content/drive/MyDrive/fundus/colored_images"
OUTPUT_DIR = "/content/drive/MyDrive/fundus/resnet18_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Dataset exists:", os.path.isdir(DATASET_PATH))
print("Output directory:", OUTPUT_DIR)

Device: cuda
GPU: Tesla T4
Dataset exists: True
Output directory: /content/drive/MyDrive/fundus/resnet18_results


In [6]:
# Load image paths and labels
class_names = sorted([
    name for name in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, name))
])

class_to_idx = {name: idx for idx, name in enumerate(class_names)}

valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

image_paths = []
labels = []

for class_name in class_names:
    class_dir = os.path.join(DATASET_PATH, class_name)

    for filename in sorted(os.listdir(class_dir)):
        file_path = os.path.join(class_dir, filename)

        if os.path.isfile(file_path) and Path(filename).suffix.lower() in valid_extensions:
            image_paths.append(file_path)
            labels.append(class_to_idx[class_name])

image_paths = np.array(image_paths)
labels = np.array(labels)

print("Total images:", len(image_paths))
print("Classes:", class_to_idx)

# 70% training, 30% temporary
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

# Split remaining 30% equally into validation and testing
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("\nSplit sizes:")
print("Training:", len(train_paths))
print("Validation:", len(val_paths))
print("Testing:", len(test_paths))
print("Total:", len(train_paths) + len(val_paths) + len(test_paths))

assert len(set(train_paths) & set(val_paths)) == 0
assert len(set(train_paths) & set(test_paths)) == 0
assert len(set(val_paths) & set(test_paths)) == 0

print("\nNo image overlap between splits.")

Total images: 3662
Classes: {'Mild': 0, 'Moderate': 1, 'No_DR': 2, 'Proliferate_DR': 3, 'Severe': 4}

Split sizes:
Training: 2563
Validation: 549
Testing: 550
Total: 3662

No image overlap between splits.


In [7]:
distribution = []

for split_name, split_labels in [
    ("Training", train_labels),
    ("Validation", val_labels),
    ("Testing", test_labels)
]:
    counts = Counter(split_labels)

    for class_name, class_idx in class_to_idx.items():
        distribution.append({
            "Split": split_name,
            "Class": class_name,
            "Images": counts.get(class_idx, 0)
        })

distribution_df = pd.DataFrame(distribution)

print(distribution_df.pivot(
    index="Class",
    columns="Split",
    values="Images"
))

print("\nSplit proportions:")
print(f"Training: {len(train_paths)/len(image_paths):.1%}")
print(f"Validation: {len(val_paths)/len(image_paths):.1%}")
print(f"Testing: {len(test_paths)/len(image_paths):.1%}")

Split           Testing  Training  Validation
Class                                        
Mild                 56       259          55
Moderate            150       699         150
No_DR               271      1263         271
Proliferate_DR       44       207          44
Severe               29       135          29

Split proportions:
Training: 70.0%
Validation: 15.0%
Testing: 15.0%


In [8]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import numpy as np

# ImageNet normalization for pretrained ResNet18
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Training: augmentation + normalization
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Validation/testing: deterministic preprocessing
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

print("Training and evaluation transforms configured!")

Training and evaluation transforms configured!


In [9]:
class FundusDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = list(image_paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert("RGB")
        label = int(self.labels[index])

        if self.transform:
            image = self.transform(image)

        return image, label


BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset = FundusDataset(
    train_paths, train_labels, train_transform
)

val_dataset = FundusDataset(
    val_paths, val_labels, eval_transform
)

test_dataset = FundusDataset(
    test_paths, test_labels, eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 81
Validation batches: 18
Testing batches: 18


In [10]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Labels in batch:", labels.tolist()[:10])

assert images.shape[1:] == (3, 224, 224)
assert torch.isfinite(images).all()

print("Data pipeline verified successfully!")

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Image dtype: torch.float32
Labels in batch: [2, 1, 1, 0, 1, 0, 2, 1, 1, 0]
Data pipeline verified successfully!


In [11]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ResNet18
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Replace the final layer with five output classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_names))

model = model.to(DEVICE)

print("Architecture: ResNet18")
print("Pretrained weights: ImageNet")
print("Input shape: [batch_size, 3, 224, 224]")
print("Output classes:", len(class_names))
print("Class mapping:", class_to_idx)
print("Device:", DEVICE)
print("\nFinal classification layer:")
print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 74.5MB/s]


Architecture: ResNet18
Pretrained weights: ImageNet
Input shape: [batch_size, 3, 224, 224]
Output classes: 5
Class mapping: {'Mild': 0, 'Moderate': 1, 'No_DR': 2, 'Proliferate_DR': 3, 'Severe': 4}
Device: cuda

Final classification layer:
Linear(in_features=512, out_features=5, bias=True)


In [12]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os

# Calculate class weights from the training split only
class_counts = torch.bincount(
    torch.tensor(train_labels, dtype=torch.long),
    minlength=len(class_names)
).float()

class_weights = len(train_labels) / (
    len(class_names) * class_counts
)

class_weights = class_weights.to(DEVICE)

# Loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer: fine-tune all ResNet18 layers
optimizer = AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# Reduce learning rate when validation loss plateaus
scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("Class names:", class_names)
print("Class counts:", class_counts.cpu().tolist())
print("Class weights:", class_weights.cpu().tolist())
print("Optimizer: AdamW")
print("Initial learning rate:", optimizer.param_groups[0]["lr"])
print("Loss function: Weighted CrossEntropyLoss")

Class names: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']
Class counts: [259.0, 699.0, 1263.0, 207.0, 135.0]
Class weights: [1.9791505336761475, 0.7333333492279053, 0.4058590531349182, 2.4763283729553223, 3.797037124633789]
Optimizer: AdamW
Initial learning rate: 0.0001
Loss function: Weighted CrossEntropyLoss


In [13]:
OUTPUT_DIR = "/content/drive/MyDrive/fundus/resnet18_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    OUTPUT_DIR, "dr_resnet18_best.pth"
)

LAST_CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR, "dr_resnet18_last.pth"
)

HISTORY_PATH = os.path.join(
    OUTPUT_DIR, "training_history.json"
)

NUM_EPOCHS = 20
PATIENCE = 5

print("Best model:", BEST_MODEL_PATH)
print("Last checkpoint:", LAST_CHECKPOINT_PATH)
print("Maximum epochs:", NUM_EPOCHS)
print("Early stopping patience:", PATIENCE)

Best model: /content/drive/MyDrive/fundus/resnet18_results/dr_resnet18_best.pth
Last checkpoint: /content/drive/MyDrive/fundus/resnet18_results/dr_resnet18_last.pth
Maximum epochs: 20
Early stopping patience: 5


In [14]:
import json
import copy
import numpy as np
from sklearn.metrics import f1_score

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": []
}

best_val_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, targets in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)

        train_loss_sum += loss.item() * batch_size
        train_correct += (outputs.argmax(1) == targets).sum().item()
        train_total += batch_size

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    val_targets_all = []
    val_predictions_all = []

    with torch.inference_mode():
        for images, targets in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, targets)

            predictions = outputs.argmax(1)
            batch_size = targets.size(0)

            val_loss_sum += loss.item() * batch_size
            val_correct += (predictions == targets).sum().item()
            val_total += batch_size

            val_targets_all.extend(targets.cpu().tolist())
            val_predictions_all.extend(predictions.cpu().tolist())

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    val_macro_f1 = f1_score(
        val_targets_all,
        val_predictions_all,
        labels=list(range(len(class_names))),
        average="macro",
        zero_division=0
    )

    scheduler.step(val_loss)

    # Record metrics
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)
    history["val_macro_f1"].append(val_macro_f1)

    # Save the best model based on validation macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": best_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_macro_f1": best_val_f1,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "history": history
            },
            BEST_MODEL_PATH
        )
    else:
        epochs_without_improvement += 1

    # Save latest checkpoint
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_macro_f1": best_val_f1,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "history": history
        },
        LAST_CHECKPOINT_PATH
    )

    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f} | "
        f"LR: {current_lr:.2e}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch+1}. "
            f"Best epoch: {best_epoch}"
        )
        break

print("\nTraining complete!")
print("Best epoch:", best_epoch)
print(f"Best validation macro F1: {best_val_f1:.4f}")
print("Best model saved to:", BEST_MODEL_PATH)

Epoch [01/20] | Train Loss: 1.1178 | Train Acc: 0.6430 | Val Loss: 0.8668 | Val Acc: 0.7978 | Val Macro F1: 0.6384 | LR: 1.00e-04
Epoch [02/20] | Train Loss: 0.8566 | Train Acc: 0.7460 | Val Loss: 0.9012 | Val Acc: 0.7486 | Val Macro F1: 0.6246 | LR: 1.00e-04
Epoch [03/20] | Train Loss: 0.7417 | Train Acc: 0.7792 | Val Loss: 0.7576 | Val Acc: 0.7851 | Val Macro F1: 0.6563 | LR: 1.00e-04
Epoch [04/20] | Train Loss: 0.6422 | Train Acc: 0.8073 | Val Loss: 0.7953 | Val Acc: 0.8069 | Val Macro F1: 0.6655 | LR: 1.00e-04
Epoch [05/20] | Train Loss: 0.5166 | Train Acc: 0.8478 | Val Loss: 0.8627 | Val Acc: 0.8215 | Val Macro F1: 0.6954 | LR: 1.00e-04
Epoch [06/20] | Train Loss: 0.4357 | Train Acc: 0.8771 | Val Loss: 0.9485 | Val Acc: 0.7851 | Val Macro F1: 0.6513 | LR: 5.00e-05
Epoch [07/20] | Train Loss: 0.3190 | Train Acc: 0.9079 | Val Loss: 0.8533 | Val Acc: 0.8087 | Val Macro F1: 0.6832 | LR: 5.00e-05
Epoch [08/20] | Train Loss: 0.2683 | Train Acc: 0.9321 | Val Loss: 0.9474 | Val Acc: 0.819

In [15]:
import torch
import numpy as np
import json
import os

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# Load the best checkpoint
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Loaded best checkpoint from epoch:", checkpoint["epoch"])
print("Best validation macro F1:", checkpoint["best_val_macro_f1"])

# Evaluate on the held-out test set
all_targets = []
all_predictions = []
all_probabilities = []

with torch.inference_mode():
    for images, targets in test_loader:
        images = images.to(DEVICE, non_blocking=True)

        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)

        all_targets.extend(targets.tolist())
        all_predictions.extend(predictions.cpu().tolist())
        all_probabilities.extend(probabilities.cpu().tolist())

# Calculate metrics
test_accuracy = accuracy_score(all_targets, all_predictions)

test_macro_f1 = f1_score(
    all_targets,
    all_predictions,
    labels=list(range(len(class_names))),
    average="macro",
    zero_division=0
)

print("\nTEST RESULTS")
print("=" * 60)
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Test Macro F1: {test_macro_f1:.4f} ({test_macro_f1 * 100:.2f}%)")

print("\nClassification Report:")
report = classification_report(
    all_targets,
    all_predictions,
    labels=list(range(len(class_names))),
    target_names=class_names,
    digits=4,
    zero_division=0
)
print(report)

print("\nConfusion Matrix:")
cm = confusion_matrix(
    all_targets,
    all_predictions,
    labels=list(range(len(class_names)))
)
print("Class order:", class_names)
print(cm)

# Save evaluation results
evaluation_results = {
    "architecture": "ResNet18",
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "best_validation_macro_f1": float(checkpoint["best_val_macro_f1"]),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "class_names": class_names,
    "classification_report": classification_report(
        all_targets,
        all_predictions,
        labels=list(range(len(class_names))),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    ),
    "confusion_matrix": cm.tolist()
}

evaluation_path = os.path.join(
    OUTPUT_DIR, "evaluation_report.json"
)

with open(evaluation_path, "w") as f:
    json.dump(evaluation_results, f, indent=2)

print("\nEvaluation report saved to:", evaluation_path)

Loaded best checkpoint from epoch: 13
Best validation macro F1: 0.7353668793081114

TEST RESULTS
Test Accuracy: 0.8455 (84.55%)
Test Macro F1: 0.7145 (71.45%)

Classification Report:
                precision    recall  f1-score   support

          Mild     0.6909    0.6786    0.6847        56
      Moderate     0.7871    0.8133    0.8000       150
         No_DR     0.9708    0.9815    0.9761       271
Proliferate_DR     0.6316    0.5455    0.5854        44
        Severe     0.5357    0.5172    0.5263        29

      accuracy                         0.8455       550
     macro avg     0.7232    0.7072    0.7145       550
  weighted avg     0.8421    0.8455    0.8434       550


Confusion Matrix:
Class order: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']
[[ 38  10   6   1   1]
 [ 14 122   2   6   6]
 [  3   2 266   0   0]
 [  0  14   0  24   6]
 [  0   7   0   7  15]]

Evaluation report saved to: /content/drive/MyDrive/fundus/resnet18_results/evaluation_report.json


In [16]:
import os

evaluation_path = os.path.join(
    OUTPUT_DIR, "evaluation_report.json"
)

print("Evaluation report exists:", os.path.exists(evaluation_path))

if os.path.exists(evaluation_path):
    with open(evaluation_path, "r") as f:
        results = json.load(f)

    print("Test accuracy:", results["test_accuracy"])
    print("Test macro F1:", results["test_macro_f1"])
else:
    print("The evaluation report has not been saved.")
    print("Rerun the complete evaluation cell.")

Evaluation report exists: True
Test accuracy: 0.8454545454545455
Test macro F1: 0.7145026233615461


In [17]:
%pip install -q onnx onnxruntime

In [18]:
import os
import json
import torch
import torch.nn as nn
import onnx

from torchvision.models import resnet18

# Paths
ONNX_PATH = os.path.join(
    OUTPUT_DIR,
    "dr_resnet18.onnx"
)

# Reconstruct the model
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

export_model = resnet18(weights=None)
export_model.fc = nn.Linear(
    export_model.fc.in_features,
    len(class_names)
)

export_model.load_state_dict(checkpoint["model_state_dict"])
export_model.eval()
export_model.cpu()

# Example input
dummy_input = torch.randn(1, 3, 224, 224)

# Export a self-contained ONNX model
torch.onnx.export(
    export_model,
    (dummy_input,),
    ONNX_PATH,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["image"],
    output_names=["logits"],
    dynamic_axes={
        "image": {0: "batch_size"},
        "logits": {0: "batch_size"}
    },
    dynamo=False
)

print("ONNX export complete!")
print("File:", ONNX_PATH)
print("File size:", round(os.path.getsize(ONNX_PATH) / (1024 * 1024), 2), "MB")

/tmp/ipykernel_3130/3382533961.py:36: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX export complete!
File: /content/drive/MyDrive/fundus/resnet18_results/dr_resnet18.onnx
File size: 42.64 MB


In [19]:
import onnx

onnx_model = onnx.load(ONNX_PATH)

# Validate the ONNX graph
onnx.checker.check_model(onnx_model)

print("ONNX graph is valid!")

print("\nModel inputs:")
for inp in onnx_model.graph.input:
    print("-", inp.name)

print("\nModel outputs:")
for out in onnx_model.graph.output:
    print("-", out.name)

print("\nONNX file exists:", os.path.exists(ONNX_PATH))
print(
    "External data references:",
    [
        entry.key
        for entry in onnx_model.metadata_props
        if entry.key == "external_data"
    ]
)

ONNX graph is valid!

Model inputs:
- image

Model outputs:
- logits

ONNX file exists: True
External data references: []


In [20]:
%pip install -q onnx onnxruntime

In [21]:
import os
import torch
import torch.nn as nn
import onnx

from torchvision.models import resnet18

# Paths
OUTPUT_DIR = "/content/drive/MyDrive/fundus/resnet18_results"
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "dr_resnet18_best.pth")
ONNX_PATH = os.path.join(OUTPUT_DIR, "dr_resnet18.onnx")

# Reconstruct the model architecture
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 5)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Export with a dynamic batch dimension
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["image"],
    output_names=["logits"],
    dynamic_axes={
        "image": {0: "batch_size"},
        "logits": {0: "batch_size"}
    },
    dynamo=False
)

print("ONNX export completed!")
print("Model:", ONNX_PATH)
print("Size:", round(os.path.getsize(ONNX_PATH) / (1024 * 1024), 2), "MB")
print("Checkpoint epoch:", checkpoint["epoch"])

/tmp/ipykernel_3130/463574356.py:29: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX export completed!
Model: /content/drive/MyDrive/fundus/resnet18_results/dr_resnet18.onnx
Size: 42.64 MB
Checkpoint epoch: 13


In [22]:
import onnx

onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

print("ONNX graph is valid!")

for inp in onnx_model.graph.input:
    print("Input:", inp.name)

for out in onnx_model.graph.output:
    print("Output:", out.name)

ONNX graph is valid!
Input: image
Output: logits


In [23]:
import numpy as np
import onnxruntime as ort
import torch

# Load ONNX Runtime
session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

# Generate a test batch
test_input = torch.randn(2, 3, 224, 224)

# PyTorch predictions
model.eval()

with torch.inference_mode():
    pytorch_logits = model(test_input).numpy()

# ONNX predictions
onnx_logits = session.run(
    ["logits"],
    {"image": test_input.numpy()}
)[0]

# Compare numerical outputs
max_difference = np.max(
    np.abs(pytorch_logits - onnx_logits)
)

pytorch_classes = pytorch_logits.argmax(axis=1)
onnx_classes = onnx_logits.argmax(axis=1)

print("PyTorch classes:", pytorch_classes)
print("ONNX classes:", onnx_classes)
print("Maximum logit difference:", max_difference)

assert np.allclose(
    pytorch_logits,
    onnx_logits,
    rtol=1e-4,
    atol=1e-5
)

assert np.array_equal(pytorch_classes, onnx_classes)

print("\nSUCCESS: ONNX predictions match PyTorch!")

PyTorch classes: [3 3]
ONNX classes: [3 3]
Maximum logit difference: 1.4305115e-06

SUCCESS: ONNX predictions match PyTorch!


In [24]:
import json

class_to_idx = {
    "Mild": 0,
    "Moderate": 1,
    "No_DR": 2,
    "Proliferate_DR": 3,
    "Severe": 4
}

metadata = {
    "model_name": "DR-ResNet18",
    "architecture": "ResNet18",
    "input_name": "image",
    "output_name": "logits",
    "input_shape": [None, 3, 224, 224],
    "input_color_format": "RGB",
    "resize": [224, 224],
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    "class_to_idx": class_to_idx,
    "idx_to_class": {
        str(idx): name
        for name, idx in class_to_idx.items()
    },
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "test_accuracy": 0.8454545454545455,
    "test_macro_f1": 0.7145026233615461
}

metadata_path = os.path.join(
    OUTPUT_DIR,
    "model_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved:", metadata_path)

Metadata saved: /content/drive/MyDrive/fundus/resnet18_results/model_metadata.json


In [25]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/fundus/resnet18_results"

required_files = [
    "dr_resnet18_best.pth",
    "dr_resnet18.onnx",
    "model_metadata.json",
    "evaluation_report.json"
]

print("RESNET18 ARTIFACT CHECK")
print("=" * 45)

for filename in required_files:
    path = os.path.join(OUTPUT_DIR, filename)

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {filename}: {size_mb:.2f} MB")
    else:
        print(f"❌ {filename}: MISSING")

RESNET18 ARTIFACT CHECK
✅ dr_resnet18_best.pth: 128.06 MB
✅ dr_resnet18.onnx: 42.64 MB
✅ model_metadata.json: 0.00 MB
✅ evaluation_report.json: 0.00 MB
